## 1. Introduction

This notebook focuses on **Step 3: Image Fine-Tuning**. In this step, we will apply fine-tuning to five top models: **MobileNetV2**, **DenseNet121**, **InceptionResNetV2**, **Xception**, and **InceptionV3**.

We will unfreeze the top layers of these models to allow them to learn the specific features of the current dataset. The models are initialized with pre-trained weights, and only the selected layers will be fine-tuned, while the rest of the layers remain frozen to retain the learned features from the original dataset.

Fine-tuning is typically done with a lower learning rate (e.g., 0.00001) to avoid making large updates that could destroy the useful pre-learned weights. Using a small learning rate ensures that the model can adapt to the new task while maintaining the integrity of the valuable knowledge it has acquired during pre-training. This strategy enables a more stable and effective fine-tuning process, minimizing the risk of overfitting and promoting generalization to unseen data.
weights.

### Optimal Number of Layers to Unfreeze:
- **MobileNetV2**: Unfreeze 4-6 layers, **max 8 layers**. This model is lightweight, and unfreezing too many layers could lead to overfitting or excessive computation.
  
- **DenseNet121**: Unfreeze 4-6 layers, **max 8 layers**. DenseNet121 has deep architecture with dense connections. Fewer layers help retain generalization power while adapting.

- **InceptionResNetV2**: Unfreeze 6-8 layers, **max 10 layers**. InceptionResNetV2 is complex, and unfreezing more layers helps capture task-specific features without losing pre-trained knowledge.

- **Xception**: Unfreeze 6-8 layers, **max 10 layers**. Xception benefits from depth and separable convolutions, allowing more layers for better adaptation to complex features.

- **InceptionV3**: Unfreeze 6-8 layers, **max 10 layers**. Similar to Xception, InceptionV3 benefits from unfreezing more layers to learn new task-specific representations.

These layer recommendations are based on practical research and the architecture characteristics of each model, designed to balance pre-trained knowledge with task-specific learning.

For this fine-tuning step, **data augmentation** is applied to enrich the training dataset and enhance model performance, similar to the previous step (Step 2). This helps to make the models more robust and capable of generalizing better to unseen data.



##  2. Import Required Libraries & Configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import os
import sys
from pathlib import Path
import importlib

# Data science and visualization libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# TensorFlow and Keras libraries for model building and image preprocessing
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# Scikit-learn metrics for model evaluation
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score

# Setup dynamic project paths
CURRENT_DIR = Path(os.getcwd()).resolve()
PROJECT_ROOT = CURRENT_DIR.parents[2]
sys.path.append(str(PROJECT_ROOT))

def get_relative_path(absolute_path):
    return str(Path(absolute_path).relative_to(PROJECT_ROOT))

print(f"Project Root Directory: {PROJECT_ROOT.name}")

# Load project config and modules from src
import config

 ## 3.  Meta data import and Preprocessing

 ### 3.1 Import Image Metadata

In [ ]:
import src.data_preprocessing.load_image_metadata
importlib.reload(src.data_preprocessing.load_image_metadata)
from src.data_preprocessing.load_image_metadata import load_image_metadata
from sklearn.utils.class_weight import compute_class_weight

# Call the function
X_train_split, X_val_split, prdtypecode_mapping = load_image_metadata()


# **************** Samples for testing  TO DEL

In [ ]:
import src.data_preprocessing.load_image_metadata
importlib.reload(src.data_preprocessing.load_image_metadata)
from src.data_preprocessing.load_image_metadata import load_image_metadata


X_train_split_full , X_val_split_full, prdtypecode_mapping = load_image_metadata()


# Nombre d'échantillons par classe à prendre (ajustez selon la taille de votre sous-échantillon)
samples_per_class = 20  # Par exemple, prendre 40 images par classe

# Sélectionner un échantillon aléatoire de chaque classe pour l'ensemble d'entraînement
X_train_split = X_train_split_full.groupby('prdtypecode_encoded', group_keys=False).apply(
    lambda x: x.sample(n=min(samples_per_class, len(x)), random_state=42)
)

# Sélectionner un échantillon aléatoire de chaque classe pour l'ensemble de validation
X_val_split = X_val_split_full.groupby('prdtypecode_encoded', group_keys=False).apply(
    lambda x: x.sample(n=min(samples_per_class, len(x)), random_state=42)
)

# Réinitialiser les index après avoir effectué l'échantillonnage
X_train_split = X_train_split.reset_index(drop=True)
X_val_split = X_val_split.reset_index(drop=True)

# Afficher les tailles des sous-échantillons
print(f"\n{'='*50}")
print(f"Train sample size: {X_train_split.shape}")
print(f"Validation sample size: {X_val_split.shape}")
print(f"\n{'='*50}")

 ### 3.2 Class Weights Calculation

In [ ]:
class_labels = np.unique(X_train_split["prdtypecode_encoded"])
class_counts = {label: np.sum(X_train_split["prdtypecode_encoded"] == label) for label in class_labels}

print('\nclass_counts:')
print(class_counts)

# Compute class weights based on distribution
class_weights = compute_class_weight('balanced', classes=class_labels, y=X_train_split["prdtypecode_encoded"])
class_weight_dict = {class_labels[i]: class_weights[i] for i in range(len(class_labels))}
class_weight_dict = {int(key): value for key, value in class_weight_dict.items()}

for label, weight in list(class_weight_dict.items())[:5]:
    print(f"Class {label}: Adjusted weight: {weight}")

 ## 4. Global Training Parameters

### 4.1 Training Parameters

In [ ]:
# Training parameters
NUM_CLASSES = 27

#Quick testing
EPOCHS = 1 
BATCH_SIZE = 32
LEARNING_RATE = 0.00001


# EPOCHS = 40
# BATCH_SIZE = 64
# LEARNING_RATE = 0.00001

class_names = [str(key) for key in prdtypecode_mapping.keys()]

in_shape_224 = (224,224,3)
in_shape_299 = (299,299,3)

MODEL_DIR = Path(config.FINE_TUNING_DIR)
STEP_NAME = 'Step3'

 ### 4.2 Data Generators for Augmentation

In [ ]:
import src.modeling_image.data_augmentation
importlib.reload(src.modeling_image.data_augmentation)
from src.modeling_image.data_augmentation import create_data_augmentation_generator


#  Data Generators for  MobileNetV2, DenseNet121 (224x224)
train_gen_224x224, valid_gen_224x224 = create_data_augmentation_generator(
    input_size=(224, 224),  # For models that use (224, 224)
    batch_size=BATCH_SIZE, 
    train_data=X_train_split, 
    validation_data=X_val_split, 
    augment=True
)

#  Data Generators for  InceptionResNetV2, Xception, InceptionV3 (299x299)
train_gen_299x299, valid_gen_299x299 = create_data_augmentation_generator(
    input_size=(299, 299),  # For models that use (299, 299)
    batch_size=BATCH_SIZE, 
    train_data=X_train_split, 
    validation_data=X_val_split, 
    augment=True
)


## 5. Model Training and Evaluation

In [ ]:

# Import necessary libraries and modules
import src.modeling_image.model_builder
importlib.reload(src.modeling_image.model_builder)
from src.modeling_image.model_builder import build_model

from src.modeling_image.cnn_training import train_model_and_save 
importlib.reload(src.modeling_image.cnn_training)
from src.modeling_image.cnn_training import train_model_and_save

from src.modeling_image.cnn_evaluation import evaluate_model 
importlib.reload(src.modeling_image.cnn_evaluation)
from src.modeling_image.cnn_evaluation import evaluate_model


from src.modeling_image.cnn_evaluation import evaluate_model 
importlib.reload(src.modeling_image.cnn_evaluation)
from src.modeling_image.cnn_evaluation import evaluate_model


importlib.reload(config)  # Reload config to ensure any updates are applied

# Notes:
# - 'model_builder.py' is used to build the model based on architecture.
# - 'cnn_training.py' contains the function to train the model and callbacks for monitoring the training process.
# - 'cnn_evaluation.py' is used for evaluating the model performance after training.
# - 'config.py' containes config settings like directories applied in the notebook.

### 5.1 Training and Evaluation for DenseNet121

#### 5.1.1 Model Building and Compilation - DenseNet121

In [ ]:
model_name = 'DenseNet121' 

# Define the number of layers to unfreeze
unfreeze_layers = 8

#Initialize the model using the build_model function.
model = build_model(
    model_name=model_name,
    input_shape=in_shape_224,
    num_classes=NUM_CLASSES,
    unfreeze_last_layers_count=unfreeze_layers  
)


#### 5.1.2 Training - DenseNet121

In [ ]:
%%time

# Calling the train_model_and_save function 
model, history = train_model_and_save(
    model=model,
    train_data=train_gen_224x224,   # The subset of training data
    val_data=valid_gen_224x224,       # The subset of validation data
    epochs=EPOCHS,               
    batch_size=BATCH_SIZE,       
    class_weight_dict=class_weight_dict,  # Apply class weights 
    learning_rate=LEARNING_RATE,          
    model_name=model_name,          # The model name for saving
    save_model_dir= MODEL_DIR,      # Directory to save the model and checkpoints
    step_name = STEP_NAME           #The current step name (e.g., 'Step2', 'Step3') for proper file naming
)


#### 5.1.3 Evaluation -DenseNet121

In [ ]:
# Call the evaluate_model function 
evaluate_model(
    model=model,
    test_data=valid_gen_224x224,  # The test data generator or dataset
    history=history,              # Pass the training history to display learning curves
    model_name=model_name,      # Name of the model
    class_names=class_names,      # List of class names (0-26)
    plot_curve=True,              # Display the learning curve (optional)
    plot_cm=True,                 # Display the confusion matrix (optional)
    report=True                   # Display the classification report (optional)
)


---

### 5.2 Training and Evaluation for MobileNetV2

#### 5.2.1 Model Building and Compilation - MobileNetV2

In [ ]:
model_name = 'MobileNetV2' 

# Define the number of layers to unfreeze
unfreeze_layers = 8

#Initialize the model using the build_model function.
model = build_model(
    model_name=model_name,
    input_shape=in_shape_224,
    num_classes=NUM_CLASSES,
    unfreeze_last_layers_count=unfreeze_layers  
)


#### 5.2.2 Training - MobileNetV2

In [ ]:
%%time

# Calling the train_model_and_save function
model, history = train_model_and_save(
    model=model,
    train_data=train_gen_224x224,   # The subset of training data
    val_data=valid_gen_224x224,       # The subset of validation data
    epochs=EPOCHS,                 
    batch_size=BATCH_SIZE,         
    class_weight_dict=class_weight_dict,  # Apply class weights 
    learning_rate=LEARNING_RATE,   
    model_name=model_name,          # The model name for saving
    save_model_dir= MODEL_DIR,      # Directory to save the model and checkpoints
    step_name = STEP_NAME         # The current step name (e.g., 'Step2', 'Step3') for proper file naming
)


#### 5.2.3 Evaluation - MobileNetV2

In [ ]:
# Call the evaluate_model function 
evaluate_model(
    model=model,
    test_data=valid_gen_224x224,  # The test data generator or dataset
    history=history,              # Pass the training history to display learning curves
    model_name=model_name,      # Name of the model
    class_names=class_names,      # List of class names (0-26)
    plot_curve=True,              # Display the learning curve (optional)
    plot_cm=True,                 # Display the confusion matrix (optional)
    report=True                   # Display the classification report (optional)
)


---

### 5.3 Training and Evaluation for InceptionResNetV2

#### 5.3.1 Model Building and Compilation - InceptionResNetV2

In [ ]:
model_name = 'InceptionResNetV2' 

# Define the number of layers to unfreeze
unfreeze_layers = 10

#Initialize the model using the build_model function.
model = build_model(
    model_name=model_name,
    input_shape=in_shape_299,
    num_classes=NUM_CLASSES,
    unfreeze_last_layers_count=unfreeze_layers  
)


#### 5.3.2 Training - InceptionResNetV2

In [ ]:
%%time

# Calling the train_model_and_save function
model, history = train_model_and_save(
    model=model,
    train_data=train_gen_299x299,   # The subset of training data
    val_data=valid_gen_299x299,       # The subset of validation data
    epochs=EPOCHS,                 
    batch_size=BATCH_SIZE,        
    class_weight_dict=class_weight_dict,  # Apply class weights 
    learning_rate=LEARNING_RATE,  
    model_name=model_name,          # The model name for saving
    save_model_dir= MODEL_DIR,      # Directory to save the model and checkpoints
    step_name = STEP_NAME         # The current step name (e.g., 'Step2', 'Step3') for proper file naming
)


#### 5.3.3 Evaluation - InceptionResNetV2

In [ ]:
# Call the evaluate_model function 
evaluate_model(
    model=model,
    test_data=valid_gen_299x299,  # The test data generator or dataset
    history=history,              # Pass the training history to display learning curves
    model_name=model_name,      # Name of the model
    class_names=class_names,      # List of class names (0-26)
    plot_curve=True,              # Display the learning curve (optional)
    plot_cm=True,                 # Display the confusion matrix (optional)
    report=True                   # Display the classification report (optional)
)


---

### 5.4 Training and Evaluation for Xception

#### 5.4.1 Model Building and Compilation - Xception

In [ ]:
model_name = 'Xception' 

# Define the number of layers to unfreeze
unfreeze_layers = 10

#Initialize the model using the build_model function.
model = build_model(
    model_name=model_name,
    input_shape=in_shape_299,
    num_classes=NUM_CLASSES,
    unfreeze_last_layers_count=unfreeze_layers  
)


#### 5.4.2 Training - Xception

In [ ]:
%%time

# Calling the train_model_and_save function 
model, history = train_model_and_save(
    model=model,
    train_data=train_gen_299x299,   # The subset of training data
    val_data=valid_gen_299x299,       # The subset of validation data
    epochs=EPOCHS,                  
    batch_size=BATCH_SIZE,         
    class_weight_dict=class_weight_dict,  # Apply class weights 
    learning_rate=LEARNING_RATE,         
    model_name=model_name,          # The model name for saving
    save_model_dir= MODEL_DIR,      # Directory to save the model and checkpoints
    step_name = STEP_NAME         # The current step name (e.g., 'Step2', 'Step3') for proper file naming
)


#### 5.4.3 Evaluation - Xception

In [ ]:
# Call the evaluate_model function 
evaluate_model(
    model=model,
    test_data=valid_gen_299x299,  # The test data generator or dataset
    history=history,              # Pass the training history to display learning curves
    model_name=model_name,      # Name of the model
    class_names=class_names,      # List of class names (0-26)
    plot_curve=True,              # Display the learning curve (optional)
    plot_cm=True,                 # Display the confusion matrix (optional)
    report=True                   # Display the classification report (optional)
)


---

### 5.5 Training and Evaluation for InceptionV3

#### 5.5.1 Model Building and Compilation - InceptionV3

In [ ]:
model_name = 'InceptionV3' 

# Define the number of layers to unfreeze
unfreeze_layers = 10

#Initialize the model using the build_model function.
model = build_model(
    model_name=model_name,
    input_shape=in_shape_299,
    num_classes=NUM_CLASSES,
    unfreeze_last_layers_count=unfreeze_layers  
)


#### 5.5.2 Training - InceptionV3

In [ ]:
%%time

# Calling the train_model_and_save function 
model, history = train_model_and_save(
    model=model,
    train_data=train_gen_299x299,   # The subset of training data
    val_data=valid_gen_299x299,       # The subset of validation data
    epochs=EPOCHS,                  
    batch_size=BATCH_SIZE,         
    class_weight_dict=class_weight_dict,  # Apply class weights 
    learning_rate=LEARNING_RATE,         
    model_name=model_name,          # The model name for saving
    save_model_dir= MODEL_DIR,      # Directory to save the model and checkpoints
    step_name = STEP_NAME         # The current step name (e.g., 'Step2', 'Step3') for proper file naming
)


#### 5.5.3 Evaluation - InceptionV3

In [ ]:
# Call the evaluate_model function 
evaluate_model(
    model=model,
    test_data=valid_gen_299x299,  # The test data generator or dataset
    history=history,              # Pass the training history to display learning curves
    model_name=model_name,      # Name of the model
    class_names=class_names,      # List of class names (0-26)
    plot_curve=True,              # Display the learning curve (optional)
    plot_cm=True,                 # Display the confusion matrix (optional)
    report=True                   # Display the classification report (optional)
)


---

---

# 6. Results and Next Steps
🔜 Transition to: `20_Image_LR_Optimization.ipynb` 

Unfreeze top layers for fine-tuning.
Adjust learning rates for better convergence.
Test advanced optimizations (batch normalization, dropout tuning...)